<img src="https://github.com/ArchiColab/sitex/blob/main/images/banner.png?raw=1" width="1000"/>

# Data Acquisition — Building Heights (Open Buildings + GBA)

This notebook downloads building-height data from two open sources, Google Open
Buildings 2.5D Temporal and the Global Building Atlas (GBA), for the same area as
`00-Data-Acquisition.ipynb`.

The notebook is optimized for Google Colab. Because it runs in the cloud, it can be used
on machines with limited computing power and in areas with limited bandwidth.

<img src="https://github.com/ArchiColab/sitex/blob/main/images/data_overview.svg?raw=1" width="1000"/>

**Library:** [`sitex`](https://github.com/ArchiColab/sitex)

**Requires:** no notebook has to run first, and no account is needed. Use the same `PLACE`
and `SLUG` as in `00-Data-Acquisition.ipynb` (see Section 3).

It is kept **separate from `00-Data-Acquisition.ipynb`** because these two downloads are
slower. **Run it before `01-ARCH-Building_Morphology.ipynb`**, which needs both files to
give every building one height (Overture's own height where it exists, then Open
Buildings, then GBA, imputed last).

**Outputs written to `data/buildings/`** (in Colab: `My Drive/Colab_Outputs/buildings/`):

| File | Source |
|---|---|
| `open_buildings_height_{year}_{slug}.tif` | Google Open Buildings 2.5D Temporal |
| `gba_{slug}.gpkg` | Global Building Atlas (GBA, TUM/zhu-xlab) |

Neither source covers every building on its own, so the building-morphology notebook
combines them instead of trusting either alone.

---
## 1. Install & import

<img src="https://github.com/ArchiColab/sitex/blob/main/images/data_setup.svg?raw=1" width="1000"/>

In [1]:
#@title <font color=#1B7192> Configuration </font>  { display-mode: "form" }

# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[data] @ git+https://github.com/ArchiColab/sitex.git"

import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/gdrive")
    %pip install "sitex[data] @ git+https://github.com/ArchiColab/sitex.git"
    from google.colab import output
    output.enable_custom_widget_manager()


Running in: Local environment


In [2]:
#@title <font color=#1B7192> Imports </font>  { display-mode: "form" }

import time
from pathlib import Path

import numpy as np
import rasterio

from sitex.data import aoi as aoi_lib
from sitex.data import open_buildings as ob
from sitex.data import gba

print("Imports OK.")


Imports OK.


## 2. Output directory

In [3]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs") if IN_COLAB else Path("..") / "data"
BLDG_DIR = DATA_DIR / "buildings"
BLDG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Buildings directory: {BLDG_DIR.resolve()}")


Buildings directory: C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\data\buildings


---
## 3. Area of Interest (same AOI as `00-Data-Acquisition.ipynb`)
<img src="https://github.com/ArchiColab/sitex/blob/main/images/data_aoi.svg?raw=1" width="1000"/>

Use **exactly the same `PLACE` (or the same drawn rectangle) and the same `SLUG`** as in
`00-Data-Acquisition.ipynb`. `SLUG` (default `pleiku`) sets the output file names, and
the building-morphology notebook looks for the height files by that name:

- **Different `SLUG`** → the building-morphology notebook stops with "file not found".
- **Same `SLUG` but a different area** → no error, but buildings outside this area get no
  measured height and are filled with imputed values instead.


In [4]:
# Select AOI method: 1 = place name (default)  |  2 = interactive map
METHOD = 1

# Method 1 — place name (validate on https://www.openstreetmap.org/ before running).
# If the name matches several boundaries, paste the OSM ID or link instead, e.g.
#   PLACE = "R7147125"   or   PLACE = "https://www.openstreetmap.org/relation/7147125"
PLACE = "Phường Pleiku, Gia Lai, Vietnam"

# Short name used in every output file name (e.g. pleiku_buildings.gpkg).
# Plain lowercase letters, no accents -- use the same SLUG in every notebook.
# Leave empty ("") to take it from the OSM name automatically (Long Xuyên -> long_xuyen).
SLUG = "pleiku"

# Method 2 / map centering
local_lat = 13.9833
local_lon = 108.0000


In [5]:
#@title <font color=#1B7192> Method 1 — AOI from place name </font>  { display-mode: "form" }
if METHOD == 1:
    print(f"Fetching OSM boundary for '{PLACE}'...")
    aoi = aoi_lib.resolve_aoi_by_place(PLACE)
    print(f"OSM boundary: {aoi.display_name}")
    local_lat, local_lon = aoi.center_lat, aoi.center_lon
    west, south, east, north = aoi.bbox
    print(f"BBox: (xmin={west:.5f}, ymin={south:.5f}, xmax={east:.5f}, ymax={north:.5f})")
    print(f"Centroid: ({aoi.center_lat:.5f}, {aoi.center_lon:.5f})")


Fetching OSM boundary for 'Phường Pleiku, Gia Lai, Vietnam'...
OSM boundary: Phường Pleiku, Pleiku, Gia Lai Province, 61111, Vietnam
BBox: (xmin=107.99506, ymin=13.96561, xmax=108.06338, ymax=14.03363)
Centroid: (13.99661, 108.03049)


In [6]:
#@title <font color=#1B7192> Method 2 — AOI from interactive map </font>  { display-mode: "form" }
if METHOD == 2:
    aoi_map, _default_bbox = aoi_lib.build_aoi_map(local_lat, local_lon, dist_m=1000, zoom=14)
    print("Draw a rectangle to customise the AOI. Yellow = default 2 km box. "
          "Run the next cell to confirm.")
    display(aoi_map)


In [7]:
#@title <font color=#1B7192> Confirm AOI </font>  { display-mode: "form" }
if METHOD == 2:
    aoi = aoi_lib.resolve_aoi_from_map(aoi_map, _default_bbox, slug="custom_aoi")
    local_lat, local_lon = aoi.center_lat, aoi.center_lon

from dataclasses import replace
if SLUG:
    aoi = replace(aoi, slug=SLUG)  # your SLUG wins over the one derived from OSM

print("\nFinal AOI (used for both sources below):")
west, south, east, north = aoi.bbox
print(f"  SW: ({south:.6f}, {west:.6f})  NE: ({north:.6f}, {east:.6f})")
print(f"  Slug (file names): {aoi.slug}")
print(f"  Auto-derived LOCAL_EPSG: {aoi.local_epsg}")



Final AOI (used for both sources below):
  SW: (13.965607, 107.995063)  NE: (14.033631, 108.063382)
  Slug (file names): pleiku
  Auto-derived LOCAL_EPSG: 32649


---
## Download datasets
<img src="https://github.com/ArchiColab/sitex/blob/main/images/data2_Download.svg?raw=1" width="1000"/>

## 4. Open Buildings 2.5D Temporal (Google)

Google's estimate of where buildings are and how tall they are, as a raster (a grid of
pixels) for the years 2016–2023. It is read from Google's public online storage; no
Earth Engine account is needed.

The dataset is split into tiles, grouped by UTM zone. Only the tiles covering your AOI are
downloaded. Tiles from the AOI's own zone are used first; tiles from a neighbouring zone
only fill pixels that are still empty and never overwrite them. The result has two bands,
`building_presence` (how likely a pixel is part of a building) and `building_height` (m),
on the AOI's local UTM grid.

**Google caps predicted heights at 100 m**, so treat any value at that limit as "tall",
not as an exact measurement.

In [8]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 4. Open Buildings 2.5D Temporal (Google) ###", flush=True)

OB_YEAR = 2023    #@param {type:"integer"}
#@markdown <font color=#1B7192> Most recent year in the 2016-2023 dataset. </font>

ob_path = BLDG_DIR / f"open_buildings_height_{OB_YEAR}_{aoi.slug}.tif"
ob.fetch_open_buildings_clip(aoi, ob_path, year=OB_YEAR)

with rasterio.open(ob_path) as src:
    ob_data = src.read()
    ob_descriptions = src.descriptions
    ob_nodata = src.nodata

presence = ob_data[ob_descriptions.index("building_presence")]
height = ob_data[ob_descriptions.index("building_height")]
confident = (presence != ob_nodata) & (height != ob_nodata) & (presence > 0.5)

print(f"\nOpen Buildings -> {ob_path}")
print(f"Confident building-presence pixels: {confident.sum():,} / {confident.size:,} "
      f"({100 * confident.sum() / confident.size:.1f}% of AOI)")
if confident.sum():
    h = height[confident]
    print(f"Height where present: min={h.min():.1f}m  median={np.median(h):.1f}m  "
          f"mean={h.mean():.1f}m  max={h.max():.1f}m")

print(f"--- 4. done in {time.time() - _t0:.2f}s ---\n")


### 4. Open Buildings 2.5D Temporal (Google) ###
2 same-zone tile(s), 1 other-zone tile(s) available as gap-fillers
  [same-zone] tile_GCCDqlJdGko.tif: filled 937,860 px
  [same-zone] tile_w_W2CFTGIyA.tif: filled 1,263,570 px
  1,362,790 / 3,564,220 px (38.2%) still empty after same-zone tiles -- filling from other-zone tiles as gap-fillers only
  [gap-fill] tile_-_6kCU08i3s.tif: filled 1,341,468 px
  21,322 px (0.60%) still empty -- no available tile covers this (a real dataset gap, not a bug)
-> (3, 1906, 1870) -> ..\data\buildings\open_buildings_height_2023_pleiku.tif

Open Buildings -> ..\data\buildings\open_buildings_height_2023_pleiku.tif
Confident building-presence pixels: 422,270 / 3,564,220 (11.8% of AOI)
Height where present: min=0.5m  median=5.0m  mean=5.7m  max=71.0m
--- 4. done in 50.49s ---



## 5. Global Building Atlas (GBA, TUM/zhu-xlab)

A worldwide set of building footprints, each with an estimated height. The data is stored
online in large tiles of 5 × 5 degrees (about 2.25 GB each); only the footprints inside
your AOI are read, never a whole tile. Each footprint is checked to lie inside the AOI
before it is kept, because GBA's own FAQ notes that some of its files carry a wrong
coordinate system.

In [9]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 5. Global Building Atlas (GBA) ###", flush=True)

gba_path = BLDG_DIR / f"gba_{aoi.slug}.gpkg"
gba_gdf = gba.fetch_gba_buildings(aoi, gba_path)

print(f"\nGBA -> {gba_path}")
print(f"{len(gba_gdf):,} buildings, height median={gba_gdf['height'].median():.2f}m  "
      f"mean={gba_gdf['height'].mean():.2f}m  max={gba_gdf['height'].max():.2f}m")

print(f"--- 5. done in {time.time() - _t0:.2f}s ---\n")


### 5. Global Building Atlas (GBA) ###
Reusing already-fetched ..\data\buildings\gba_pleiku.gpkg

GBA -> ..\data\buildings\gba_pleiku.gpkg
71,834 buildings, height median=3.57m  mean=3.58m  max=27.96m
--- 5. done in 0.56s ---



---
## 6. Summary
<img src="https://github.com/ArchiColab/sitex/blob/main/images/data_Export.svg?raw=1" width="1000"/>

In [10]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

print(f"{'='*55}")
print("  Building Heights Acquisition Summary")
print(f"{'='*55}")
print(f"  AOI                : {aoi.slug}  (LOCAL_EPSG={aoi.local_epsg})")
print(f"  Open Buildings     : {ob_path}  ({100 * confident.sum() / confident.size:.1f}% confident coverage)")
print(f"  GBA                : {gba_path}  ({len(gba_gdf):,} buildings)")
print("\nNext: 01-ARCH-Building_Morphology.ipynb reconciles a single height per "
      "building (Overture height where available, then Open Buildings, then GBA, "
      "imputation last) and classifies building function.")


  Building Heights Acquisition Summary
  AOI                : pleiku  (LOCAL_EPSG=32649)
  Open Buildings     : ..\data\buildings\open_buildings_height_2023_pleiku.tif  (11.8% confident coverage)
  GBA                : ..\data\buildings\gba_pleiku.gpkg  (71,834 buildings)

Next: 01-ARCH-Building_Morphology.ipynb reconciles a single height per building (Overture height where available, then Open Buildings, then GBA, imputation last) and classifies building function.
